In [34]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [35]:
data=pd.read_csv('digit-recognizer/train.csv')


In [36]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 251.5 MB


In [37]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [38]:
data=np.array(data)
m,n=data.shape
np.random.shuffle(data)
data_dev=data[0:1000].T
X_dev=data_dev[1:n]
Y_dev=data_dev[0]
data_train=data[1000:m].T
Y_train=data_train[0]
X_train=data_train[1:n]


In [39]:
Y_train

array([9, 5, 1, ..., 4, 8, 2])

In [61]:
def int_params():
    w1=np.random.randn(10,784)-0.5
    b1=np.random.randn(10,1)-0.5
    w2=np.random.randn(10,10)-0.5
    b2=np.random.randn(10,1)-0.5
    return w1,b1,w2,b2

In [41]:
def ReLu(Z):
    return np.maximum(0,Z)#greater than zero return 'Z' less than 0 return 0

In [42]:
def softmax(Z):
    return np.exp(Z)/np.sum(np.exp(Z))# apply exponental on every element and divide that with the vertical sum of columns that is now collapsed into a single row.

In [43]:
def forward_prop(w1,b1,w2,b2,x):
    Z1=w1.dot(x) +b1
    A1=ReLu(Z1)
    Z2=w2.dot(A1)+b2
    A2=softmax(Z2)# sofmax returns probabilites
    return Z1,A1,Z2,A2


In [44]:
def one_hot(Y):
    one_hot_Y=np.zeros((Y.size,Y.max()+1))
    one_hot_Y[np.arange(Y.size),Y]=1
    one_hot_Y=one_hot_Y.T
    return one_hot_Y

In [45]:
def der_ReLu(Z):#derivative of relu
    return Z>0

In [57]:
def back_prop(Z1,A1,Z2,A2,W2,X,Y):
    one_hot_Y = one_hot(Y)                # expected shape (n_y, m)
    m = one_hot_Y.shape[1]                # use m from one-hot (robust)

    # output layer gradients
    dZ2 = A2 - one_hot_Y                  # (n_y, m)
    dW2 = (1.0 / m) * dZ2.dot(A1.T)       # (n_y, n_h)
    dB2 = (1.0 / m) * np.sum(dZ2, axis=1, keepdims=True)  # (n_y, 1)

    # hidden layer gradients
    dZ1 = W2.T.dot(dZ2) * der_ReLu(Z1)    # (n_h, m)
    dW1 = (1.0 / m) * dZ1.dot(X.T)        # (n_h, n_x)
    dB1 = (1.0 / m) * np.sum(dZ1, axis=1, keepdims=True)  # (n_h, 1)

    # optional sanity checks (comment out in production)
    # assert dB1.shape[1] == 1 and dB2.shape[1] == 1
    # assert dW2.shape == W2.shape

    return dW1, dB1, dW2, dB2

In [47]:
def update_params(w1,b1,w2,b2,dW1,dB1,dW2,dB2,alpha):
    w1=w1-alpha*dW1
    b1=b1-alpha*dB1
    w2=w2-alpha*dW2
    b2=b2-alpha*dB2
    return w1,b1,w2,b2

In [48]:
def get_predictions(A2):
    return np.argmax(A2,0)

In [49]:
def get_accuracy(predictions,Y):
    print(predictions,Y)
    return np.sum(predictions==Y)/Y.size

In [62]:
def gradient_desent(X,Y,iterations,alpha):
    w1,b1,w2,b2=int_params()
    
    for i in range(iterations):
        Z1, A1, Z2,A2=forward_prop(w1,b1,w2,b2,X)
        dw1,db1,dw2,dwb2=back_prop(Z1, A1, Z2,A2,w2,X,Y)
       
        w1,b1,w2,b2=update_params(w1,b1,w2,b2,dw1,db1,dw2,dwb2,alpha)
        
        if i%10==0:
            print("iterations=",i)
            print("accuracy=",get_accuracy(get_predictions(A2),Y))
    return w1,b1,w2,b2



In [63]:
 w1,b1,w2,b2=gradient_desent(X_train,Y_train,100,.1)

C:\Users\ajayj\AppData\Local\Temp\ipykernel_21856\1054677053.py:2: RuntimeWarning: overflow encountered in exp
  return np.exp(Z)/np.sum(np.exp(Z))# apply exponental on every element and divide that with the vertical sum of columns that is now collapsed into a single row.
C:\Users\ajayj\AppData\Local\Temp\ipykernel_21856\1054677053.py:2: RuntimeWarning: invalid value encountered in divide
  return np.exp(Z)/np.sum(np.exp(Z))# apply exponental on every element and divide that with the vertical sum of columns that is now collapsed into a single row.


iterations= 0
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 10
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 20
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 30
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 40
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 50
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 60
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 70
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 80
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
iterations= 90
[0 0 0 ... 0 0 0] [9 5 1 ... 4 8 2]
accuracy= 0.09873170731707318
